# 04 — repo-TPE + градиентный вес w(x) (метод 2)

Wrapper над исходным ноутбуком (эксперимент `04_tpe_wx_gradient`). Исходник не изменяется.

> **Замечание о воспроизводимости.** Эти wrapper-ноутбуки запускают исходные `fin_*.ipynb` *как есть*.
> Для запуска нужны: (1) сами исходные `.ipynb`, (2) папка-репозиторий `tpe/` в Google Drive
> (`drive/MyDrive/content/tpe`), (3) `ConfigSpace==1.2.0`, `parzen-estimator`, `optuna`, `scipy`.
> Пакет `tpe` **не входит** в git-репозиторий курсовой — он лежит только в Drive. Поэтому без Drive код не воспроизводится.
> Параметры (SEEDS=range(30), MAX_EVALS=100, BASE_KWARGS) **зашиты внутри исходников**: wrapper их не переопределяет,
> чтобы не нарушать правило 'не менять исходный код'. Унификация здесь — это единый запуск, единое хранение и единый сбор метрик.

## Разбор эксперимента

**Цель:** проверить мягкую модификацию repo-TPE множителем w(x), зависящим от нормы аналитического градиента.

**Способ изменения ТПЕ:** `TPEOptimizer(..., weight_fn=w)`; w(x)=clip(1+0.2·z, 0.8, 1.2), 4 формы (smooth/smooth_inv/sign_like/sign_like_inv).

**🔴🔴 ПОДТВЕРЖДЁННЫЙ ДЕФЕКТ (проверено по исходнику `tpe/optimizer/tpe.py`, строки 232–252).**
repo-`TPEOptimizer` вызывает `weight_fn` один раз на пачку кандидатов с 1D-массивом
`samples = np.stack([x0_cands, x1_cands], axis=1).mean(axis=1)` — это **среднее координат (x0+x1)/2**,
а НЕ полные 2D-точки и НЕ покоординатно. Затем `_as_2d_points` в ноутбуке делает из скаляра m точку **(m, 0)**
и считает там аналитический ∇f → **градиент берётся в фиктивной точке**. Для Sphere истинная ‖∇f‖=2√(x0²+x1²),
а код фактически считает |x0+x1|. Значит **метод 2 в текущем виде НЕ реализует градиентное взвешивание TPE.**

**Доп. деталь:** `compute_log_weight`→`scale_to_range` заново min-max-нормирует веса в [0.8,1.2] по кандидатам →
множитель `1+0.2·z` отбрасывается, выживает только РАНГ grad-нормы-в-(m,0). Нормировка ∇ — константа, поэтому raw и norm для grad_* совпадают.

**Что МОЖНО (строго):** grad_*-результаты нельзя интерпретировать как градиентное взвешивание; эффект мал и не осмыслен (~0.006–0.02, как no_w).
**Что НЕЛЬЗЯ:** утверждать пользу/вред w(x) по градиенту — метод сначала надо исправить (передавать в weight_fn 2D-точки).

Ячейка ниже воспроизводит логику вызова `weight_fn` из repo-TPE на маленьком примере — наглядно показывает дефект.

In [ ]:
# Демонстрация дефекта: как repo-TPE на самом деле зовёт weight_fn (см. tpe/optimizer/tpe.py:232-252).
import numpy as np

# Кандидаты по двум осям (как config_cands['x0'], ['x1']):
x0 = np.array([3.0, -4.0, 1.0]); x1 = np.array([4.0, 3.0, -1.0])
# repo-TPE формирует ОДИН 1D-массив = среднее координат на кандидата:
samples = np.stack([x0, x1], axis=1).mean(axis=1)
print('samples (mean coords) =', samples)   # -> [3.5 -0.5 0.0]

# Ноутбук (_as_2d_points) превратит каждый скаляр m в точку (m, 0):
fake_points = np.column_stack([samples, np.zeros_like(samples)])
real_points = np.column_stack([x0, x1])
sphere_grad_norm = lambda P: 2.0*np.linalg.norm(P, axis=1)
print('grad-norm в фиктивной (m,0):', sphere_grad_norm(fake_points))
print('grad-norm в РЕАЛЬНОЙ (x0,x1):', sphere_grad_norm(real_points))
print('=> величины разные -> метод 2 считает не тот градиент')


In [ ]:
# === Единая настройка путей и окружения (общая для всех wrapper-ноутбуков) ===
# ВАЖНО: этот блок НЕ трогает исходные ноутбуки. Он только готовит окружение,
# монтирует Drive и определяет, где лежат (а) исходные .ipynb и (б) папка-репозиторий tpe.
import os, sys, json, shutil, subprocess, time, glob
from pathlib import Path

# 1) Монтируем Google Drive (в Colab). Локально просто пропустится.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', repr(e))

# 2) Где лежат ИСХОДНЫЕ ноутбуки (5 файлов fin_*.ipynb).
#    Поменяйте при необходимости на свой путь в Drive.
ORIGINALS_DIR = Path(os.environ.get('ORIGINALS_DIR', 'drive/MyDrive/content/k5_originals'))
if not ORIGINALS_DIR.exists():
    for cand in ['/content/drive/MyDrive/content/k5_originals',
                 'drive/MyDrive/content', '/content/drive/MyDrive/content', '.']:
        if list(Path(cand).glob('fin_*.ipynb')):
            ORIGINALS_DIR = Path(cand); break

# 3) Корень анализа (эта папка coursework_analysis).
ANALYSIS_ROOT = Path(os.environ.get('ANALYSIS_ROOT', '.')).resolve()
for cand in [Path('.'), Path('coursework_analysis'), Path('/content/coursework_analysis')]:
    if (cand / 'configs' / 'common_config.json').exists():
        ANALYSIS_ROOT = cand.resolve(); break

RAW_DIR   = ANALYSIS_ROOT / 'results' / 'raw'
PROC_DIR  = ANALYSIS_ROOT / 'results' / 'processed'
TABLES_DIR= ANALYSIS_ROOT / 'tables'
FIG_DIR   = ANALYSIS_ROOT / 'figures'
for d in (RAW_DIR, PROC_DIR, TABLES_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

COMMON = json.load(open(ANALYSIS_ROOT / 'configs' / 'common_config.json'))
EXPS   = json.load(open(ANALYSIS_ROOT / 'configs' / 'experiments_config.json'))
EXP_BY_ID = {e['id']: e for e in EXPS['experiments']}

print('ORIGINALS_DIR =', ORIGINALS_DIR)
print('ANALYSIS_ROOT =', ANALYSIS_ROOT)
print('originals found:', sorted(p.name for p in ORIGINALS_DIR.glob('fin_*.ipynb')))

In [ ]:
# === Единый запуск исходного ноутбука БЕЗ его изменения + сбор результатов ===
# Стратегия: исходные ноутбуки самодостаточны (ставят зависимости, монтируют Drive,
# импортируют repo-tpe, считают и СОХРАНЯЮТ свои CSV в ./results/*.csv).
# Мы выполняем их 'как есть' через nbconvert --execute (копия в /tmp, оригинал не трогаем),
# затем переносим произведённые CSV в coursework_analysis/results/raw/ под унифицированными именами.

def run_original_and_harvest(exp_id, fix_seed=0, timeout=7200):
    exp = EXP_BY_ID[exp_id]
    src = ORIGINALS_DIR / exp['source_notebook']
    assert src.exists(), f'Не найден исходный ноутбук: {src}'

    work = Path('/content/_run') if Path('/content').exists() else Path('/tmp/_run')
    work.mkdir(parents=True, exist_ok=True)
    tmp_nb = work / src.name
    shutil.copy(src, tmp_nb)

    # Лог метаданных запуска (seed/versions). Сам seed зашит в исходниках (SEEDS=range(30)).
    meta = {'exp_id': exp_id, 'source': str(src), 'fix_seed_requested': fix_seed,
            'started': time.strftime('%Y-%m-%d %H:%M:%S')}
    try:
        import numpy, optuna, ConfigSpace
        meta['versions'] = {'numpy': numpy.__version__,
                            'optuna': getattr(optuna,'__version__','?'),
                            'ConfigSpace': getattr(ConfigSpace,'__version__','?')}
    except Exception as e:
        meta['versions_error'] = repr(e)

    print(f'Executing {src.name} ... (может занять минуты)')
    cmd = ['jupyter','nbconvert','--to','notebook','--execute',
           '--ExecutePreprocessor.timeout=%d'%timeout, str(tmp_nb),
           '--output', tmp_nb.stem + '_executed']
    res = subprocess.run(cmd, cwd=str(work), capture_output=True, text=True)
    meta['nbconvert_returncode'] = res.returncode
    if res.returncode != 0:
        print('STDERR tail:\n', res.stderr[-2000:])
    # Исходники сохраняют CSV в <cwd>/results/. cwd = work.
    produced = list((work / 'results').glob('*.csv')) if (work/'results').exists() else []
    harvested = []
    for produced_name, target_name in zip(exp['produces_csv'], exp['harvest_as']):
        p = work / produced_name
        if p.exists():
            shutil.copy(p, RAW_DIR / target_name)
            harvested.append(target_name)
    meta['harvested'] = harvested
    meta['finished'] = time.strftime('%Y-%m-%d %H:%M:%S')
    json.dump(meta, open(RAW_DIR / f'{exp_id}_runmeta.json','w'), ensure_ascii=False, indent=2)
    print('Harvested ->', harvested)
    return meta

In [ ]:
EXP_ID = '04_tpe_wx_gradient'
exp = EXP_BY_ID[EXP_ID]
print('Source notebook:', exp['source_notebook'])
print('TPE modification:', exp['tpe_modification'])
print('Variants:', exp['variants'])

### Запуск (раскомментируйте, когда Drive и пакет `tpe` доступны)

In [ ]:
# meta = run_original_and_harvest(EXP_ID)
# meta

### Быстрый просмотр собранной сводки

In [ ]:
import pandas as pd
for name in exp['harvest_as']:
    p = RAW_DIR / name
    if p.exists() and p.suffix=='.csv':
        print('\n===', name, '===')
        display(pd.read_csv(p).head(20))
    else:
        print('нет файла (ещё не запускали):', name)